In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.5 MB/s eta 0:00:00


In [ ]:
# Load env variables
from anthropic import Anthropic
from google.colab import userdata
from IPython.display import Markdown



In [ ]:
# 1. Retrieve the key from Colab Secrets
my_anthropic_key   = userdata.get('ANTHROPIC_API_KEY')


my_anthropic_model = userdata.get('CLAUDE_MODEL')


In [ ]:

# Create an API client
client = Anthropic(
    api_key=my_anthropic_key,
)

print("Anthropic client initialized successfully!")


Anthropic client initialized successfully!


In [ ]:
# define a system prompt
#system_prompt = """
#You are a creative screenwriter for feature films.
#"""
system_prompt = None

In [ ]:
# Utility functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def add_assistant_message_with_response_content(messages, response):
    assistant_message = {"role": "assistant", "content": response.content}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None, tools=None):
    params = {
        "model": my_anthropic_model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    # Add tools if provided
    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)

    # Return the full message object instead of hardcoding content[0].text
    # This allows your calling code to check if Claude returned text or a tool call.
    return message

In [ ]:
# Tools and Schemas
from anthropic.types import ToolParam
from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})


In [ ]:
def get_tool_use_block(response):
    """
    Safely searches through Claude's response and returns the first ToolUseBlock.
    Returns None if no tool was called.
    """
    for block in response.content:
        if block.type == 'tool_use':
            return block

    return None

In [ ]:
# Make a request

# Start with an empty message list
messages = []

messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = chat(messages, tools=[get_current_datetime_schema])
add_assistant_message_with_response_content(messages, response)

result = get_current_datetime(**get_tool_use_block(response).input)

add_user_message(
    messages,
    [{
        "type": "tool_result",
        "tool_use_id": get_tool_use_block(response).id,
        "content": "15:04:22",
        "is_error": False
    }]
)

messages


[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_012SB4ADndPdpTiPXj4LuTat', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_012SB4ADndPdpTiPXj4LuTat',
    'content': '15:04:22',
    'is_error': False}]}]